# Evaluate model states

A position is a dictionary of selected variable or node names and values.
A model state also contains the cached values and update flags of its
computational nodes. Use `model` from
{doc}`tutorials/notebooks/11-model-building` below.

## Try a position

In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import tensorflow_probability.substrates.jax.distributions as tfd

import liesel.model as lsl

In [2]:
# Python cell fragment: imports are visible in each guide.
x_data = jnp.linspace(-1.0, 1.0, 80)
y_data = 1.0 + 2.0 * x_data + 0.7 * jax.random.normal(jax.random.key(42), (80,))

prior_scale = lsl.Var.new_value(2.5, name="prior_scale")
beta = lsl.Var.new_param(
    jnp.zeros(2),
    dist=lsl.Dist(tfd.Normal, loc=0.0, scale=prior_scale),
    name="beta",
)
variance = lsl.Var.new_param(
    1.0,
    dist=lsl.Dist(tfd.LogNormal, loc=0.0, scale=0.5),
    name="variance",
)
x = lsl.Var.new_obs(x_data, name="x")

mu = lsl.Var.new_calc(
    lambda x, b: b[0] + b[1] * x,
    x,
    beta,
    name="mu",
)
sigma = lsl.Var.new_calc(jnp.sqrt, variance, name="sigma")
y = lsl.Var.new_obs(
    y_data,
    dist=lsl.Dist(tfd.Normal, loc=mu, scale=sigma),
    name="y",
)
model = lsl.Model(y)

# Match the tutorial after its reactive update.
beta.value = jnp.array([1.0, 2.0])


In [3]:
position = model.extract_position(["beta", "variance"])
position["beta"] = jnp.array([0.5, 1.5])

model.update()
candidate = model.update_state(position, model.state)

In [4]:
model.extract_position(["beta"], candidate)

{'beta': Array([0.5, 1.5], dtype=float32)}

In [5]:
model.extract_position(["beta"])

{'beta': Array([1., 2.], dtype=float32)}

The first result contains the candidate coefficients; the second contains the
model's existing coefficients. `update_state` leaves the model unchanged by
default. Its input state must already be up to date: it only triggers updates
through the supplied position. Call {meth}`update() <liesel.model.Var.update>` first if automatic updates
have been disabled. Weak variables normally remain calculated, read-only values.

To apply the candidate explicitly:

In [6]:
model.state = candidate

This is also how you can apply a position returned by an optimizer. See
{doc}`optimizer-customization` for a fit-to-fit example and
{meth}`~liesel.model.Model.update_state` for the in-place option.

## Calculate gradients

In [7]:
log_prob = lsl.LogProb(model)
position = model.extract_position(["beta", "variance"])

In [8]:
round(float(log_prob(position)), 3)

-102.532

In [9]:
gradient = log_prob.grad(position)

In [10]:
{name: np.asarray(value).round(3) for name, value in gradient.items()}

{'beta': array([40.275,  8.956], dtype=float32),
 'variance': np.float32(-16.079)}

The gradient has the same dictionary keys and parameter shapes as `position`.
It describes the local change in the unnormalized log density, not a fitted
parameter value. Use `component="log_lik"` or `"log_prior"` to inspect
one contribution, and {class}`~liesel.model.FlatLogProb` for an interface that
accepts a flat parameter array.

## Write JAX calculations

Pass every changing dependency into the calculation:

In [11]:
def linear_mean(x, beta):
    return beta[0] + beta[1] * x


predictor = lsl.Var.new_calc(
    linear_mean,
    model.vars["x"],
    model.vars["beta"],
    name="prediction_check",
)

In [12]:
np.asarray(predictor.value[:3]).round(3)

array([-1.   , -0.962, -0.924], dtype=float32)

Use JAX array operations in calculations that will be differentiated or compiled.
Avoid reading mutable globals or capturing {attr}`beta.value <liesel.model.Var.value>` in a closure: those
values would not be explicit graph inputs. A pure function depends only on its
inputs and has no side effects.

Python control flow over static values can work. Branching on a traced array
value needs JAX control flow such as `jax.lax.cond` or an elementwise
`jnp.where`. Keep data preparation outside the graph when it need not be
re-evaluated, and test any calculation that must also run on new prediction data.

By default, calculator variables cache their results and recompute when their
inputs change. `new_calc(..., cache=False)` avoids storing the result, but
repeats the calculation on access. Consider it for a cheap reshaping operation, not an
expensive matrix decomposition. See {meth}`~liesel.model.Var.new_calc` for details.